# 21 — Client Intelligence Monitor
> Give it a company name and get back a structured brief: recent funding, executive moves, regulatory exposures, strategic signals, and prioritised account actions — all in one call.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core langgraph pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import Literal

from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from pydantic import BaseModel, Field


# --- Data shapes ---

class FundingEvent(BaseModel):
    round_type: str = Field(description="Funding round type, e.g. Series B, debt facility.")
    amount_usd_m: float = Field(description="Amount raised in USD millions.")
    date: str = Field(description="Approximate date, e.g. Q2 2024.")
    lead_investor: str = Field(description="Lead investor name, or 'undisclosed'.")


class LeadershipChange(BaseModel):
    role: str = Field(description="Job title affected, e.g. CFO, CTO.")
    change_type: Literal["hire", "departure", "promotion"] = Field(
        description="Nature of the leadership change."
    )
    name: str = Field(description="Person's name, or 'undisclosed'.")
    date: str = Field(description="Approximate date.")


class RegulatoryExposure(BaseModel):
    topic: str = Field(description="Regulatory area, e.g. GDPR, antitrust, emissions.")
    severity: Literal["low", "medium", "high"] = Field(
        description="Estimated business impact severity."
    )
    summary: str = Field(description="One-sentence description of the exposure.")


class StrategicSignal(BaseModel):
    signal: str = Field(
        description="Plain-English description of a strategic move or intent signal."
    )
    implication: str = Field(
        description="Why this matters for relationship or opportunity planning."
    )


class ClientIntelBrief(BaseModel):
    company: str = Field(description="Company name.")
    funding_events: list[FundingEvent] = Field(
        description="Recent funding rounds or capital events."
    )
    leadership_changes: list[LeadershipChange] = Field(
        description="Recent C-suite or senior leadership changes."
    )
    regulatory_exposures: list[RegulatoryExposure] = Field(
        description="Material regulatory risks or obligations."
    )
    strategic_signals: list[StrategicSignal] = Field(
        description="Strategic moves, partnerships, or stated priorities."
    )
    relationship_actions: list[str] = Field(
        description="Concrete recommended next steps for the account team, highest priority first."
    )


# --- Research sources (swap these for live APIs: Exa, SEC EDGAR, LinkedIn, CRM) ---

_NEWS: dict[str, str] = {
    "acme corp": (
        "Acme Corp raised a $120M Series C led by Tiger Global in Q1 2024. "
        "The company also announced a strategic partnership with Microsoft Azure."
    ),
    "beta industries": (
        "Beta Industries announced a $300M debt facility in Q4 2023. "
        "Revenue grew 28% YoY per their investor day presentation."
    ),
    "nova logistics": (
        "Nova Logistics closed a $75M Series B led by Andreessen Horowitz in Q3 2024. "
        "The company is accelerating its European rollout and hired 200 staff in Q4."
    ),
}

_FILINGS: dict[str, str] = {
    "acme corp": "Acme Corp is under FTC scrutiny for potential antitrust issues related to its 2023 acquisition of Streamline Inc.",
    "beta industries": "Beta Industries disclosed material climate-related reporting obligations under new SEC rules effective 2025.",
    "nova logistics": "Nova Logistics received a GDPR enforcement notice from the Irish DPA in Q2 2024 related to cross-border driver data transfers.",
}

_LEADERSHIP: dict[str, str] = {
    "acme corp": "CFO Sarah Lin departed in March 2024. New CFO David Park hired from Goldman Sachs, effective May 2024.",
    "beta industries": "CTO Michael Chen promoted to Chief AI Officer, a newly created role, in January 2024.",
    "nova logistics": "CEO James Howell stepped down in August 2024. COO Priya Sharma appointed interim CEO while a permanent search is underway.",
}

_MARKET: dict[str, str] = {
    "acme corp": "Acme Corp publicly stated intent to expand into the APAC market by end of 2024 and is hiring aggressively in Singapore.",
    "beta industries": "Beta Industries filed three patents in Q1 2024 related to autonomous logistics, signalling an AI-first product pivot.",
    "nova logistics": "Nova Logistics signed a five-year contract with a major European retailer and is evaluating acquisition targets in the last-mile delivery space.",
}


@tool
def search_news(company: str) -> str:
    """Search recent press coverage for a company."""
    return _NEWS.get(company.lower(), f"No significant recent news found for {company}.")


@tool
def search_regulatory_filings(company: str) -> str:
    """Search regulatory filings and compliance actions for a company."""
    return _FILINGS.get(company.lower(), f"No material regulatory filings found for {company}.")


@tool
def search_leadership_changes(company: str) -> str:
    """Search for recent executive and senior leadership changes."""
    return _LEADERSHIP.get(company.lower(), f"No recent leadership changes found for {company}.")


@tool
def search_market_signals(company: str) -> str:
    """Search for strategic signals: patents, partnerships, geographic expansion, product shifts."""
    return _MARKET.get(company.lower(), f"No significant market signals found for {company}.")


# --- Agent ---

_SYSTEM = SystemMessage(
    """You are a client intelligence analyst. Use the research tools to gather
signals on the target company from multiple sources, then summarise all
findings in a single comprehensive brief.

Steps:
1. Call search_news for recent press coverage.
2. Call search_regulatory_filings for any regulatory activity.
3. Call search_leadership_changes for executive moves.
4. Call search_market_signals for strategic intent signals.
5. Summarise all findings in plain English."""
)


def run(company: str) -> ClientIntelBrief:
    """Build a structured intelligence brief for the given company."""
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    tools = [search_news, search_regulatory_filings, search_leadership_changes, search_market_signals]
    agent = create_react_agent(llm, tools, prompt=_SYSTEM)

    result = agent.invoke({"messages": [{"role": "user", "content": f"Build an intelligence brief for: {company}"}]})
    last = result["messages"][-1].content

    structured_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(ClientIntelBrief)
    return structured_llm.invoke(
        f"Convert the following intelligence findings into a ClientIntelBrief object.\n\nCompany: {company}\n\nFindings:\n{last}"
    )

## The scenario

Nova Logistics is a fast-growing freight-tech company your team has been tracking as a target account. They just closed a Series B, their CEO stepped down mid-year, and there is an active GDPR enforcement action in Europe. The agent pulls all of that together and tells your account team exactly what to do next.

In [ ]:
brief = run("Nova Logistics")

print(f"\n{'=' * 60}")
print(f"  INTEL BRIEF — {brief.company.upper()}")
print(f"{'=' * 60}")

if brief.funding_events:
    print("\nFUNDING")
    for f in brief.funding_events:
        print(f"  {f.date}  {f.round_type}  ${f.amount_usd_m:.0f}M  (led by {f.lead_investor})")

if brief.leadership_changes:
    print("\nLEADERSHIP MOVES")
    for lc in brief.leadership_changes:
        print(f"  {lc.date}  {lc.role}  [{lc.change_type.upper()}]  {lc.name}")

if brief.regulatory_exposures:
    print("\nREGULATORY EXPOSURE")
    for r in brief.regulatory_exposures:
        print(f"  [{r.severity.upper()}]  {r.topic}")
        print(f"          {r.summary}")

if brief.strategic_signals:
    print("\nSTRATEGIC SIGNALS")
    for s in brief.strategic_signals:
        print(f"  * {s.signal}")
        print(f"    WHY IT MATTERS: {s.implication}")

print("\nACCOUNT ACTIONS (priority order)")
for i, action in enumerate(brief.relationship_actions, 1):
    print(f"  {i}. {action}")

print()

## Use your own data

Replace the input above with:
- **your target company name** — any company your team is tracking or managing
- **your own data sources** — swap the mock `_NEWS`, `_FILINGS`, `_LEADERSHIP`, and `_MARKET` dicts for live connectors (Exa web search, SEC EDGAR, a CRM API, or a LinkedIn feed)

The agent will return the same structured brief — funding events, leadership moves, regulatory flags, strategic signals, and prioritised next steps — regardless of where the data comes from.